# Download and preprocess dbNSFP Variant Files

Convert the variant .gz files to parquet files for faster processing and merge with our ClinVar annotations

In [17]:
import os
import polars as pl
import pandas as pd
# import concurrent.futures

## Get the prediction scores from dbNSFP

Using scripts/data_processing.sh to download data from https://drive.google.com/file/d/1HNoZUswadC1ODavU7AUvm0DZ13jKVFKW/view

In [ ]:
## Test reading in a chr variant file
# chr_num = "22"
# df_chr = pl.read_csv(f'../dbnsfp_data/dbNSFP5.0/dbNSFP5.0a/dbNSFP5.0a_variant.chr{chr_num}.gz', 
#                     separator='\t',
#                     n_threads=16,
#                     infer_schema_length=100000,
#                     ignore_errors=True,
#                     truncate_ragged_lines=True, 
#                     # n_rows=5,
#                     low_memory=False)
# df_chr_par = pl.scan_parquet(f"../dbnsfp_data/dbNSFP5.0/dbNSFP5.0a/dbNSFP5.0a_variant.chr{chr_num}.parquet")
# df_chr_par = df_chr_par.filter(pl.col("clinvar_id")!='.')

## Convert the chr variant files to parquet

__For easier accessing and processing__

In [6]:
# def convert_to_parquet(chr_num):
#     file_path = f"../1_inputs/raw_data/dbNSFP5.0a/dbNSFP5.0a_variant.chr{chr_num}.gz"
#     df = pl.read_csv(file_path, 
#                      separator='\t',
#                      n_threads=16,
#                      infer_schema_length=1000000,
#                      ignore_errors=True,
#                      truncate_ragged_lines=True,
#                      low_memory=False)
#     df = df.with_columns(
#         pl.col("aaref").replace(aa_single_to_three).alias("aaref_3"),
#         pl.col("aaalt").replace(aa_single_to_three).alias("aaalt_3"),
#     )
#     parquet_path = f"../1_inputs/raw_data/dbNSFP5.0a/dbNSFP5.0a_variant.chr{chr_num}.parquet"
#     df.write_parquet(parquet_path)
#     return parquet_path

# chr_num_list = [chr_file.split('.')[-2].split('chr')[-1] for chr_file in os.listdir('../1_inputs/raw_data/dbNSFP5.0a') if "chr" in chr_file and chr_file.endswith(".gz")]

# # Use ThreadPoolExecutor for parallel processing
# with concurrent.futures.ThreadPoolExecutor() as executor:
#     futures = [executor.submit(convert_to_parquet, chr_num) for chr_num in chr_num_list]
    
#     # Collect the results
#     parquet_files = [future.result() for future in concurrent.futures.as_completed(futures)]

# print("Conversion to Parquet completed.")

## Get the original ClinVar Annotations

Merging the most up-to-date clinvar annotations to dbNSFP for consistency

In [4]:
processed_clinvar_parq = "../../1_allele_collection/3_outputs/processed_data/clinvar/clinvar_processed.parquet"
clinvar = pl.scan_parquet(processed_clinvar_parq)

In [13]:
clinvar.select(pl.col("Type")).unique().collect().to_numpy()

array([['Microsatellite'],
       ['single nucleotide variant'],
       ['Insertion'],
       ['Variation'],
       ['copy number loss'],
       ['Indel'],
       ['Deletion'],
       ['Complex'],
       ['copy number gain'],
       ['Translocation'],
       ['Duplication'],
       ['Inversion']], dtype=object)

In [14]:
clinvar.collect().shape

(3390885, 47)

In [10]:
clinvar_snv = clinvar.filter(pl.col("Type")=='single nucleotide variant') ##.select(pl.col("Type")).unique().collect().to_numpy()
clinvar_snv.collect().shape

(3105660, 47)

In [15]:
clinvar_snv.collect()

#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),nsv/esv (dbVar),RCVaccession,PhenotypeIDS,PhenotypeList,Origin,OriginSimple,Assembly,ChromosomeAccession,Chromosome,Start,Stop,Cytogenetic,ReviewStatus,NumberSubmitters,Guidelines,TestedInGTR,OtherIDs,SubmitterCategories,VariationID,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification,clinvar_nt_change,clinvar_aa_change,SPDI,RefSeq_mRNA,StarStatus,clinvar_clnsig_clean
i64,str,str,i64,str,str,cat,i64,str,i64,str,str,str,str,str,str,cat,str,str,i64,i64,str,cat,i64,str,str,str,i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,cat
15043,"""single nucleotide variant""","""NM_014630.3(ZNF592):c.3136G>A …",9640,"""ZNF592""","""HGNC:28986""","""Uncertain significance""",0,"""Jun 29, 2015""",150829393,"""-""","""RCV000000014""","""MONDO:MONDO:0033005,MedGen:C45…","""Galloway-Mowat syndrome 1""","""germline""","""germline""","""GRCh38""","""NC_000015.10""","""15""",84799209,84799209,"""15q25.3""","""no assertion criteria provided""",1,"""-""","""N""","""ClinGen:CA210674,UniProtKB:Q92…",1,4,84799209,"""G""","""A""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV000020157""","""-""","""-""","""3136G>A ""","""Gly1046Arg""","""NC_000015.10:84799209:G:A""","""NM_014630.3""",null,"""4_VUS"""
15044,"""single nucleotide variant""","""NM_017547.4(FOXRED1):c.694C>T …",55572,"""FOXRED1""","""HGNC:26927""","""Pathogenic""",1,"""Mar 12, 2024""",267606829,"""-""","""RCV000000015|RCV000578659|RCV0…","""MONDO:MONDO:0032624,MedGen:C47…","""Mitochondrial complex 1 defici…","""germline""","""germline""","""GRCh38""","""NC_000011.10""","""11""",126275389,126275389,"""11q24.2""","""criteria provided, multiple su…",6,"""-""","""N""","""ClinGen:CA113792,OMIM:613622.0…",3,5,126275389,"""C""","""T""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV000680696|SCV001363290|SCV0…","""-""","""-""","""694C>T ""","""Gln232Ter""","""NC_000011.10:126275389:C:T""","""NM_017547.4""",2,"""1_Pathogenic"""
15045,"""single nucleotide variant""","""NM_017547.4(FOXRED1):c.1289A>G…",55572,"""FOXRED1""","""HGNC:26927""","""Likely pathogenic""",1,"""Jun 06, 2024""",267606830,"""-""","""RCV000000016""","""MONDO:MONDO:0032624,MedGen:C47…","""Mitochondrial complex 1 defici…","""germline""","""germline""","""GRCh38""","""NC_000011.10""","""11""",126277517,126277517,"""11q24.2""","""criteria provided, single subm…",2,"""-""","""N""","""ClinGen:CA113794,UniProtKB:Q96…",3,6,126277517,"""A""","""G""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV005680614""","""-""","""-""","""1289A>G ""","""Asn430Ser""","""NC_000011.10:126277517:A:G""","""NM_017547.4""",1,"""1_Pathogenic"""
15046,"""single nucleotide variant""","""NM_025152.3(NUBPL):c.166G>A (p…",80224,"""NUBPL""","""HGNC:20278""","""Conflicting classifications of…",1,"""Nov 12, 2024""",200401432,"""-""","""RCV000196589|RCV000622708|RCV0…","""MedGen:C3661900|MeSH:D030342,M…","""not provided|Inborn genetic di…","""germline;paternal""","""germline""","""GRCh38""","""NC_000014.9""","""14""",31562125,31562125,"""14q12""","""criteria provided, conflicting…",5,"""-""","""N""","""OMIM:613621.0001,ClinGen:CA321…",2,214885,31562125,"""G""","""A""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV000251971|SCV000742093|SCV0…","""-""","""-""","""166G>A ""","""Gly56Arg""","""NC_000014.9:31562125:G:A""","""NM_025152.3""",1,"""3_Conflicting"""
15048,"""single nucleotide variant""","""NM_000410.4(HFE):c.845G>A (p.C…",3077,"""HFE""","""HGNC:4886""","""Pathogenic/Pathogenic, low pen…",1,"""Mar 04, 2025""",1800562,"""-""","""RCV000000019|RCV000210820|RCV0…","""MONDO:MONDO:0021001,MedGen:C34…","""Hemochromatosis type 1|Heredit…","""biparent

In [22]:
clinvar_snv.filter(pl.col("VariationID")==2333487).collect()

#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),nsv/esv (dbVar),RCVaccession,PhenotypeIDS,PhenotypeList,Origin,OriginSimple,Assembly,ChromosomeAccession,Chromosome,Start,Stop,Cytogenetic,ReviewStatus,NumberSubmitters,Guidelines,TestedInGTR,OtherIDs,SubmitterCategories,VariationID,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification,clinvar_nt_change,clinvar_aa_change,SPDI,RefSeq_mRNA,StarStatus,clinvar_clnsig_clean
i64,str,str,i64,str,str,cat,i64,str,i64,str,str,str,str,str,str,cat,str,str,i64,i64,str,cat,i64,str,str,str,i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,cat
2324965,"""single nucleotide variant""","""NM_199261.4(TPTE):c.37G>C (p.V…",7179,"""TPTE""","""HGNC:12023""","""Uncertain significance""",0,"""Aug 02, 2021""",-1,"""-""","""RCV004175219""","""MedGen:CN169374""","""not specified""","""germline""","""germline""","""GRCh38""","""NC_000021.9""","""21""",10541137,10541137,"""21p11.2""","""criteria provided, single subm…",1,"""-""","""N""","""ClinGen:CA9978559""",2,2333487,10541137,"""G""","""C""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV003666233""","""-""","""-""","""37G>C ""","""Val13Leu""","""NC_000021.9:10541137:G:C""","""NM_199261.4""",1,"""4_VUS"""


In [ ]:
chr_num_list = [chr_file.split('.')[-2].split('chr')[-1] for chr_file in os.listdir('./raw_inputs/dbNSFP5.0a/') if "chr" in chr_file and chr_file.endswith(".gz")]
clin_var_dfs = []
for chr_num in chr_num_list:
    df_chr_par = pl.scan_parquet(f"./raw_inputs/dbNSFP5.0a/dbNSFP5.0a_variant.chr{chr_num}.parquet")
    df_chr_par_clinvar = df_chr_par.filter(pl.col("clinvar_id")!='.')
    clinvar_per_chr = clinvar_snv.filter(pl.col("Chromosome")==chr_num)#.collect()
    clinvar_per_chr = clinvar_per_chr.with_columns(pl.col("VariationID").cast(pl.String).alias("clinvar_id"))
    df_chr_par_clinvar = df_chr_par_clinvar.join(clinvar_per_chr, on="clinvar_id", how="left")
    df_chr_par_clinvar = df_chr_par_clinvar.collect()
    clin_var_dfs.append(df_chr_par_clinvar)

StringCacheMismatchError: cannot compare categoricals coming from different sources, consider setting a global StringCache.

Help: if you're using Python, this may look something like:

    with pl.StringCache():
        # Initialize Categoricals.
        df1 = pl.DataFrame({'a': ['1', '2']}, schema={'a': pl.Categorical})
        df2 = pl.DataFrame({'a': ['1', '3']}, schema={'a': pl.Categorical})
    # Your operations go here.
    pl.concat([df1, df2])

Alternatively, if the performance cost is acceptable, you could just set:

    import polars as pl
    pl.enable_string_cache()

on startup.

In [ ]:
df_clinvar = pl.concat(clin_var_dfs, how="vertical_relaxed")

# parquet_path = f"../dbnsfp_data/dbNSFP5.0/dbNSFP5.0a/dbNSFP5.0a_variant.clin_var_annot.parquet"
# df_clinvar.write_parquet(parquet_path)

StringCacheMismatchError: cannot compare categoricals coming from different sources, consider setting a global StringCache.

Help: if you're using Python, this may look something like:

    with pl.StringCache():
        # Initialize Categoricals.
        df1 = pl.DataFrame({'a': ['1', '2']}, schema={'a': pl.Categorical})
        df2 = pl.DataFrame({'a': ['1', '3']}, schema={'a': pl.Categorical})
    # Your operations go here.
    pl.concat([df1, df2])

Alternatively, if the performance cost is acceptable, you could just set:

    import polars as pl
    pl.enable_string_cache()

on startup.